<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/02-data-structures/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: Solutions

Try the exercises first. Each solution comes with notes on why it uses the data structure it does, which is the main skill this module is about.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below. You don't need to read it,
# but feel free to look at the test cases if you're stuck.

import copy
import math


class Raises:
    """Marks a test case where the function is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=60):
    """repr() that stays readable for long lists."""
    text = repr(x)
    if len(text) <= limit:
        return text
    if isinstance(x, tuple):
        return "(" + ", ".join(_short(item, limit) for item in x) + ")"
    if isinstance(x, list):
        head = ", ".join(repr(item) for item in x[:3])
        return f"[{head}, ... {len(x)} items]"
    return text[:limit - 3] + "..."


def _call_str(func, args):
    return f"{func.__name__}({', '.join(_short(a) for a in args)})"


def _run(func, cases):
    for args, expected, hint in cases:
        call = _call_str(func, args)
        before = copy.deepcopy(args)
        try:
            result = func(*args)
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {call}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {expected!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {call}")
            print(f"      expected {expected!r}, but it returned {result!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if args != before:
            print(f"FAIL  {call}")
            print("      your function modified its input. It should leave it unchanged.")
            return
        if not _same(result, expected):
            print(f"FAIL  {call}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {func.__name__}")


def check_unique_in_order(f):
    _run(f, [
        (([3, 1, 3, 2, 1],), [3, 1, 2], "keep the order of first appearance. A set doesn't remember order."),
        ((["b", "a", "b"],), ["b", "a"], None),
        (([],), [], None),
        (([1, 1, 1],), [1], None),
    ])


def check_invert_dict(f):
    _run(f, [
        (({"cat": 0, "dog": 1},), {0: "cat", 1: "dog"}, None),
        (({},), {}, None),
        (({"a": 1, "b": 1},), Raises(ValueError), "two keys share the value 1, so the inverted dict would lose one of them"),
    ])


def check_merge_configs(f):
    _run(f, [
        (({"lr": 0.001, "epochs": 10}, {"lr": 0.01}), {"lr": 0.01, "epochs": 10}, None),
        (({"a": 1}, {}), {"a": 1}, None),
        (({}, {"b": 2}), {"b": 2}, None),
        (({"a": 1, "b": 2}, {"b": 3, "c": 4}), {"a": 1, "b": 3, "c": 4}, None),
    ])


def check_oov_words(f):
    _run(f, [
        ((["the", "cat", "sat"], ["the", "dog", "sat", "dog"]), ["dog"], "each word should appear once"),
        ((["a", "b"], ["c", "b", "a", "d"]), ["c", "d"], None),
        ((["x"], ["x", "x"]), [], None),
        (([], ["b", "a"]), ["a", "b"], "return the words sorted"),
    ])


def check_group_by_label(f):
    _run(f, [
        (([([1, 2], "cat"), ([3, 4], "dog"), ([5, 6], "cat")],),
         {"cat": [[1, 2], [5, 6]], "dog": [[3, 4]]}, None),
        (([],), {}, None),
        (([("a", 1), ("b", 1), ("c", 2)],), {1: ["a", "b"], 2: ["c"]}, None),
    ])


def check_top_k(f):
    scores = {"cat": 0.6, "dog": 0.3, "fox": 0.1}
    _run(f, [
        ((scores, 2), [("cat", 0.6), ("dog", 0.3)], None),
        ((scores, 5), [("cat", 0.6), ("dog", 0.3), ("fox", 0.1)], "if k is bigger than the number of classes, return all of them"),
        (({"b": 0.5, "a": 0.5, "c": 0.2}, 2), [("a", 0.5), ("b", 0.5)], "break ties alphabetically by label"),
        (({"x": 0.9}, 0), [], None),
    ])


def check_word_frequencies(f):
    _run(f, [
        (("The cat saw the other cat.",), {"the": 2, "cat": 2, "saw": 1, "other": 1}, "lowercase everything and strip punctuation from the ends of words"),
        (("Free! Free money, free.", 2), {"free": 3}, None),
        (("",), {}, None),
        (("Hello , world",), {"hello": 1, "world": 1}, "a lone ',' becomes an empty string after stripping. Skip it."),
    ])


def check_deep_get(f):
    _run(f, [
        (({"model": {"lr": 0.01}}, "model.lr"), 0.01, None),
        (({"model": {"lr": 0.01}}, "model.momentum"), None, None),
        (({"model": {"lr": 0.01}}, "optimizer.lr", 0.001), 0.001, None),
        (({"a": {"b": {"c": 5}}}, "a.b.c"), 5, None),
        (({"a": {"b": 3}}, "a.b.c", "missing"), "missing", "config['a']['b'] is 3, not a dict, so there's nothing to look inside"),
        (({"lr": 0.0}, "lr", 1.0), 0.0, "0.0 is a real value, not a missing one"),
        (({"a": {"b": None}}, "a.b", 7), None, "the key exists and its value is None, so return None, not the default"),
    ])


def check_records_to_columns(f):
    _run(f, [
        (([{"a": 1, "b": 2}, {"a": 3, "b": 4}],), {"a": [1, 3], "b": [2, 4]}, None),
        (([{"a": 1}, {"b": 2}],), {"a": [1, None], "b": [None, 2]}, "fill missing values with None"),
        (([],), {}, None),
        (([{"x": "p", "y": 0.5}],), {"x": ["p"], "y": [0.5]}, None),
    ])


def check_stratified_split(f):
    s1 = [(1, "a"), (2, "b"), (3, "a"), (4, "a"), (5, "b"), (6, "a")]
    s3 = [(i, "x") for i in range(10)]
    s4 = [(i, "spam" if i in (2, 7) else "ham") for i in range(10)]
    _run(f, [
        ((s1, 0.5), ([(1, "a"), (2, "b"), (3, "a")], [(4, "a"), (5, "b"), (6, "a")]), None),
        ((s1, 0.0), (s1, []), "with test_ratio=0 nothing goes to test. Careful: idx[-0:] is the whole list"),
        ((s3, 0.2), (s3[:8], s3[8:]), None),
        ((s4, 0.5),
         ([(0, "ham"), (1, "ham"), (2, "spam"), (3, "ham"), (4, "ham")],
          [(5, "ham"), (6, "ham"), (7, "spam"), (8, "ham"), (9, "ham")]),
         "split each label separately, then keep both outputs in the original order"),
    ])


def check_most_similar(f):
    docs = {"d1": {"cat", "sat", "mat"}, "d2": {"dog", "sat", "log"},
            "d3": {"cat", "mat", "hat"}, "d4": set()}
    _run(f, [
        (({"cat", "mat"}, docs, 2), ["d1", "d3"], "d1 and d3 tie, so sort them by name"),
        (({"dog", "log"}, docs, 1), ["d2"], None),
        (({"cat", "mat"}, docs, 10), ["d1", "d3", "d2", "d4"], None),
        ((set(), {"a": set(), "b": {"x"}}, 2), ["a", "b"], "two empty sets have an empty union. Treat their similarity as 0.0"),
    ])


def check_bigram_model(f):
    _run(f, [
        ((["the cat sat", "the cat ran"],), {"the": {"cat": 2}, "cat": {"sat": 1, "ran": 1}}, "count pairs inside each sentence only. 'sat' is not followed by 'the'"),
        ((["a b a b"],), {"a": {"b": 2}, "b": {"a": 1}}, None),
        ((["hello"],), {}, "a single word has no next word"),
        ((["I am", "am I"],), {"i": {"am": 1}, "am": {"i": 1}}, "lowercase the words, and don't count pairs across two sentences"),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `unique_in_order`

In [2]:
def unique_in_order(items):
    return list(dict.fromkeys(items))

In [3]:
check_unique_in_order(unique_in_order)

PASS  all 4 tests passed for unique_in_order


Dictionaries keep insertion order and can't have duplicate keys, so `dict.fromkeys(items)` does all the work in one step. The values are all `None`, but we only care about the keys.

The more explicit version uses a set to remember what you've already seen:

```python
def unique_in_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result
```

Notice that `seen` is a set, not a list. If you wrote `if item not in result`, every check would scan the whole result list, and the function would get very slow on big inputs. That's the lesson from section 5 in practice.

### Exercise 2: `invert_dict`

In [4]:
def invert_dict(d):
    inverted = {value: key for key, value in d.items()}
    if len(inverted) != len(d):
        raise ValueError("values are not unique, so the dict can't be inverted")
    return inverted

In [5]:
check_invert_dict(invert_dict)

PASS  all 3 tests passed for invert_dict


The comprehension builds the inverted dict, and if any value appeared twice, a later key overwrote an earlier one, so the result has fewer entries. Comparing the lengths catches that without needing a separate check.

Without the length check, `{"a": 1, "b": 1}` would silently become `{1: "b"}`, and `"a"` would vanish. Silent data loss is exactly the kind of bug that's hard to track down later, so raising an error is the right call.

### Exercise 3: `merge_configs`

In [6]:
def merge_configs(defaults, overrides):
    return {**defaults, **overrides}

In [7]:
check_merge_configs(merge_configs)

PASS  all 4 tests passed for merge_configs


`{**defaults, **overrides}` unpacks both dicts into a new one, left to right, so later keys overwrite earlier ones. `defaults | overrides` does the same thing in Python 3.9+.

The tempting version is:

```python
defaults.update(overrides)
return defaults
```

It returns the right answer, but it changes `defaults` permanently. If you call it again for a second experiment, that experiment starts from the first one's settings. The check catches this because it compares the inputs before and after the call.

### Exercise 4: `oov_words`

In [8]:
def oov_words(train_tokens, test_tokens):
    return sorted(set(test_tokens) - set(train_tokens))

In [9]:
check_oov_words(oov_words)

PASS  all 4 tests passed for oov_words


Set difference does exactly what the question asks, and converting to sets also removes the duplicates. `sorted` accepts any iterable and returns a list, so there's no need to convert back to a list first.

Tracking the OOV rate (the fraction of test tokens that are OOV) is a useful sanity check in NLP. If it's high, your vocabulary doesn't cover the test data well, maybe because the test data comes from a different domain, like training on news articles and testing on tweets.

## Medium

### Exercise 5: `group_by_label`

In [10]:
from collections import defaultdict


def group_by_label(samples):
    groups = defaultdict(list)
    for features, label in samples:
        groups[label].append(features)
    return dict(groups)

In [11]:
check_group_by_label(group_by_label)

PASS  all 3 tests passed for group_by_label


`defaultdict(list)` creates an empty list the first time each label is seen, so the loop body is a single line. Converting to a plain `dict` at the end is optional, but it's a good habit: it stops the caller from accidentally creating new keys just by reading from it.

Grouping by class is useful for things like computing per-class statistics, balancing a dataset by sampling the same number from each class, or the stratified split you'll write in exercise 10.

### Exercise 6: `top_k`

In [12]:
def top_k(scores, k):
    ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
    return ranked[:k]

In [13]:
check_top_k(top_k)

PASS  all 4 tests passed for top_k


The key `(-score, label)` sorts by score from high to low, then by label from A to Z for ties. That's the tuple-key trick from section 1.

Slicing handles the "k is too large" case for free: `ranked[:10]` on a 3-item list just gives you all 3.

`heapq.nlargest` is faster when you want a few items out of a huge number, but the alphabetical tie-break is awkward with it, since it picks the *largest* keys and we want ties to go to the *smallest* label. With a few hundred classes, sorting everything is perfectly fast anyway.

### Exercise 7: `word_frequencies`

In [14]:
from collections import Counter


def word_frequencies(text, min_count=1):
    words = (word.strip(".,!?;:") for word in text.lower().split())
    counts = Counter(word for word in words if word)
    return {word: n for word, n in counts.items() if n >= min_count}

In [15]:
check_word_frequencies(word_frequencies)

PASS  all 4 tests passed for word_frequencies


`str.strip(chars)` removes any of the given characters from both ends of the string, so `"cat.".strip(".,!?;:")` is `"cat"` and `",".strip(".,!?;:")` is `""`. The `if word` filter drops those empty strings.

The first line is a generator expression, so no intermediate list is built. It feeds straight into `Counter`. The last line filters by `min_count` with a dict comprehension.

This is a very simple tokenizer. It doesn't handle things like apostrophes ("don't"), hyphens or punctuation in the middle of a word ("e.g."). Real tokenizers deal with all of that, but this is enough for the spam filter in the mini project.

### Exercise 8: `deep_get`

In [16]:
def deep_get(config, path, default=None):
    current = config
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

In [17]:
check_deep_get(deep_get)

PASS  all 7 tests passed for deep_get


The loop walks down one level per key. Before each step it checks two things: that the current value is a dictionary at all, and that the key is in it. If either check fails, we stop and return the default.

The tempting shortcut is to chain `.get()` calls and use `or default` at the end. That breaks on the last two test cases: `0.0 or default` gives the default, because `0.0` is falsy. Same for `None`. Checking with `in` asks "does this key exist?", which is the actual question, instead of "is this value truthy?". It's the `is None` lesson from Module 1 again, in a different form.

Config libraries like OmegaConf and Hydra, which are popular for ML experiments, let you access nested config values with exactly this kind of dotted path.

### Exercise 9: `records_to_columns`

In [18]:
def records_to_columns(records):
    keys = list(dict.fromkeys(key for record in records for key in record))
    return {key: [record.get(key) for record in records] for key in keys}

In [19]:
check_records_to_columns(records_to_columns)

PASS  all 4 tests passed for records_to_columns


This is done in two passes. The first collects every key that appears in any record, in first-seen order, using the `dict.fromkeys` trick from exercise 1. The second builds one list per key, using `.get(key)` so missing values come out as `None`.

Doing it in one pass is tricky, because when you meet a new key in the third record, you'd have to go back and fill in `None` for the first two. Two simple passes are easier to get right than one clever one.

This is the same conversion pandas does when you call `pd.DataFrame(records)` on a list of dicts. JSON from APIs usually comes as records, while DataFrames, NumPy and most ML libraries work with columns. Missing values become `NaN` in pandas, which Module 7 covers.

## Hard

### Exercise 10: `stratified_split`

In [20]:
from collections import defaultdict


def stratified_split(samples, test_ratio):
    positions = defaultdict(list)
    for i, (_, label) in enumerate(samples):
        positions[label].append(i)

    test_positions = set()
    for label, idx in positions.items():
        n_test = round(len(idx) * test_ratio)
        test_positions.update(idx[len(idx) - n_test:])

    train = [s for i, s in enumerate(samples) if i not in test_positions]
    test = [s for i, s in enumerate(samples) if i in test_positions]
    return train, test

In [21]:
check_stratified_split(stratified_split)

PASS  all 4 tests passed for stratified_split


The trick is to work with *positions* instead of the samples themselves.

1. Group the positions (indexes) of the samples by label.
2. For each label, pick the last `n_test` positions and add them to a set.
3. Go through the samples once more in their original order and send each one to train or test depending on whether its position is in the set.

Because the last step walks through the original list, both outputs keep the original order automatically. And because `test_positions` is a set, each `i in test_positions` check is O(1).

Note `idx[len(idx) - n_test:]` instead of `idx[-n_test:]`. It's the same `-0` trap from Module 1's `train_test_split`: when `n_test` is 0, `idx[-0:]` is the whole list.

In real projects you'd shuffle within each class first. scikit-learn's `train_test_split(..., stratify=y)` does this for you.

### Exercise 11: `most_similar`

In [22]:
def jaccard(a, b):
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def most_similar(query, documents, k):
    ranked = sorted(documents, key=lambda name: (-jaccard(query, documents[name]), name))
    return ranked[:k]

In [23]:
check_most_similar(most_similar)

PASS  all 4 tests passed for most_similar


Pulling `jaccard` out into its own function keeps each piece simple and testable on its own. The ranking reuses the `(-score, name)` tuple key from `top_k`.

`sorted(documents, ...)` iterates over the dict's keys, so we get a sorted list of names directly.

Jaccard similarity is used for real in near-duplicate detection, for example finding copies of the same web page in a crawled dataset before training a language model. Comparing every pair of documents is too slow at that scale, so a technique called MinHash estimates Jaccard similarity quickly. Modern retrieval for things like RAG uses embedding vectors and cosine similarity instead of word sets, but the "score everything, take the top k" structure is the same.

### Exercise 12: `bigram_model`

In [24]:
from collections import Counter, defaultdict


def bigram_model(sentences):
    model = defaultdict(Counter)
    for sentence in sentences:
        words = sentence.lower().split()
        for current, following in zip(words, words[1:]):
            model[current][following] += 1
    return {word: dict(counts) for word, counts in model.items()}

In [25]:
check_bigram_model(bigram_model)

PASS  all 4 tests passed for bigram_model


`zip(words, words[1:])` is a neat idiom for walking through consecutive pairs: it pairs each word with the one after it, and stops when `words[1:]` runs out, so there's no index arithmetic and no off-by-one errors.

`defaultdict(Counter)` gives every new word its own empty counter, so `model[current][following] += 1` just works. The last line converts everything to plain dicts.

Once you have the model, predicting the most likely next word is one line:

```python
max(model["the"], key=model["the"].get)
```

Below, we use it to generate text by repeatedly picking a likely next word. It's not good text, but it's the same basic loop that large language models run: given the words so far, pick the next one, add it, repeat. The difference is that an LLM looks at thousands of previous tokens instead of one, and uses a neural network instead of a count table.

In [26]:
import random

corpus = [
    "the model learns from the data",
    "the data is split into batches",
    "the model is trained on batches of data",
    "the loss goes down as the model learns",
]
model = bigram_model(corpus)

def generate(model, start, length=8, seed=0):
    rng = random.Random(seed)
    words = [start]
    for _ in range(length):
        options = model.get(words[-1])
        if not options:
            break
        choices, weights = zip(*options.items())
        words.append(rng.choices(choices, weights=weights)[0])
    return " ".join(words)

for seed in range(3):
    print(generate(model, "the", seed=seed))

the loss goes down as the model is split
the model is trained on batches of data is
the loss goes down as the data is split


`rng.choices(choices, weights=weights)` picks a next word at random, but in proportion to how often it was seen. Always picking the single most common word instead (called *greedy decoding*) tends to get stuck in loops like "the model is the model is...". Sampling, and settings like *temperature* that control how random the sampling is, exist in LLMs for exactly this reason.

## What's next

The mini project in `04_mini_project.ipynb` builds a Naive Bayes spam filter from scratch. It's built almost entirely out of `Counter`, `defaultdict` and sets.